# GovIntel Decoding Experiments

Phase 7 decoding sweep for the fine-tuned Mistral model. This notebook tests temperature, top-k, and top-p settings, then scores each response for entity accuracy and faithfulness.

In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import InferenceClient

from govintel.training.decoding import decoding_grid, score_decoding_output, select_best_config

REPO_ROOT = Path.cwd()
TEST_QUERIES_PATH = REPO_ROOT / "eval" / "test_queries.json"
GOLD_ANSWERS_PATH = REPO_ROOT / "eval" / "gold_answers.json"
RESULTS_DIR = REPO_ROOT / "eval" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = os.environ.get("HF_MODEL_ID", "avisheksaha/govintel-mistral-7b")
HF_API_TOKEN = os.environ.get("HF_API_TOKEN", "")

In [ ]:
configs = decoding_grid(
    temperatures=(0.0, 0.3, 0.7),
    top_ks=(50, 100),
    top_ps=(0.9, 0.95),
)
configs

In [ ]:
def load_eval_cases(limit: int = 5) -> list[dict[str, object]]:
    test_queries = json.loads(TEST_QUERIES_PATH.read_text())
    gold_answers = json.loads(GOLD_ANSWERS_PATH.read_text())
    cases = []
    for query in test_queries[:limit]:
        case_id = query["id"]
        case = dict(query)
        case["gold_answer"] = gold_answers[case_id]
        cases.append(case)
    return cases


eval_cases = load_eval_cases(limit=5)
eval_cases[0]

In [ ]:
client = InferenceClient(model=MODEL_ID, token=HF_API_TOKEN or None)

def build_prompt(case: dict[str, object]) -> str:
    return (
        "Write a concise federal procurement intelligence brief. "
        "Use the provided gold evaluation context only.\n\n"
        f"Question: {case['question']}\n"
        f"Agency filter: {case['agency_filter']}\n"
        f"Context: {case['gold_answer']}\n"
        f"Valid citations: {', '.join(case['valid_contract_ids'])}"
    )


def generate_answer(case: dict[str, object], config) -> str:
    response = client.chat_completion(
        messages=[{"role": "user", "content": build_prompt(case)}],
        max_tokens=600,
        temperature=config.temperature,
        top_p=config.top_p,
        extra_body={"top_k": config.top_k},
    )
    return response.choices[0].message.content.strip()

In [ ]:
all_results = []
for config in configs:
    scored_cases = []
    for case in eval_cases:
        answer = generate_answer(case, config)
        scored = score_decoding_output(
            config,
            answer=answer,
            expected_contractors=case["expected_contractors"],
            contexts=[case["gold_answer"]],
            valid_contract_ids=case["valid_contract_ids"],
        )
        scored_cases.append(scored)
    entity_accuracy = sum(result.entity_accuracy for result in scored_cases) / len(scored_cases)
    faithfulness = sum(result.faithfulness for result in scored_cases) / len(scored_cases)
    all_results.append(
        {
            "config": config.name,
            "temperature": config.temperature,
            "top_k": config.top_k,
            "top_p": config.top_p,
            "entity_accuracy": entity_accuracy,
            "faithfulness": faithfulness,
            "overall_score": (entity_accuracy + faithfulness) / 2,
        }
    )

RESULTS_PATH = RESULTS_DIR / "decoding_experiments.json"
RESULTS_PATH.write_text(json.dumps(all_results, indent=2))
all_results

In [ ]:
ranked = sorted(all_results, key=lambda row: row["overall_score"], reverse=True)
best_row = ranked[0]

# Keep a typed best-configuration check in the notebook so it exercises the same helper used in tests.
best_result = select_best_config([
    score_decoding_output(
        config,
        answer=" ".join(case["valid_contract_ids"]),
        expected_contractors=case["expected_contractors"],
        contexts=[case["gold_answer"]],
        valid_contract_ids=case["valid_contract_ids"],
    )
    for config in configs[:1]
    for case in eval_cases[:1]
])

best_row, best_result.config